# N15. Structured erasure decoding, arm H, paired comparisons, and the results workbook

What N14 established on development data (deterministic training, B and M identical):

* No domain shift: every decoder scores 99.3 to 99.9% on clean markers.
* Visible cell damage is local. Under grey and white, cells touching the occluder are often
  confidently wrong; cells further away are as good as clean. This matches the decoder's
  12 pixel receptive field (one cell plus one neighbour each side).
* Capping confidences does not fix soft decoding for the published decoder.
* Arm F (erasure output, random brightness training) flags 98 to 99% of hidden grey and white
  cells but only 1 to 3% of hidden black cells. Every model is 31 to 39 points below its
  oracle on black, so the black failure is about not knowing which cells are hidden.
* All Welch comparisons across seeds were inconclusive with 5 seeds.

What this notebook does:

1. **Arm H**: arm F trained only with black occluders. If even H cannot flag hidden black
   cells, black occlusion cannot be detected from the cells alone. The other 35 runs load
   from the N14 checkpoints; only H trains (about 15 minutes).
2. **Structured erasure decoding**: try 21 occlusion hypotheses (no occlusion, or a strip 1
   to 5 cells wide from each of the 4 sides). For each hypothesis the hidden cells are
   ignored and every id is scored with soft decoding, minus a penalty lambda per ignored
   cell. The best id and hypothesis are chosen together. lambda is tuned on videos 1 and 2;
   every reported number is from videos 3 to 5. Tested also on clean markers (must not hurt)
   and on centre occluders, where the strip assumption is wrong.
3. **Paired comparisons**: with deterministic training, arms with the same number of outputs
   start from the same weights and see batches in the same order (B, E, M, G share this;
   so do C, D, F, H). Within a group, seeds can be compared one to one, which removes most
   seed noise. Pairing is never used across groups.
4. **Results workbook**: every saved result file is collected into one Excel workbook.

**Before running: Runtime, Restart session, then Run all.** Development data only; video 6
stays untouched.


In [1]:
!pip -q install opencv-contrib-python==4.10.0.84 h5py 2>&1 | tail -1

In [2]:
%%writefile deeparuco_port.py
"""
Port the pretrained DeepArUco++ decoder from Keras .h5 to PyTorch, without TensorFlow.

The published decoder (models/dec_new.h5 in AVAuco/deeparuco) is a Keras 2.11 model:

    Input(32, 32, 1)
    Conv2D(8, 5x5, valid) -> PReLU -> MaxPool2D(2)
    Conv2D(4, 3x3, valid) -> PReLU -> MaxPool2D(2)
    Conv2D(1, 1x1, valid, sigmoid)   ->  (6, 6, 1)

Only h5py is needed to read the weights, so the repo's Python 3.9 + TensorFlow
environment is never required for baseline inference.

IMPORTANT PORTING DETAIL
    Keras PReLU defaults to shared_axes=None, so alpha is PER-ELEMENT, not per-channel:
    the stored alphas have shape (28, 28, 8) and (12, 12, 4). torch.nn.PReLU only
    supports per-channel alpha, so using it here silently produces a different model.
    SpatialPReLU below keeps the full alpha tensor.

Layout conversions:
    Keras conv kernel (kh, kw, in, out) -> torch (out, in, kh, kw)
    Keras PReLU alpha (h, w, c)         -> torch (c, h, w)
    Keras 'valid' padding               -> torch padding=0

Usage:
    from deeparuco_port import load_weights, NumpyDecoder, TorchDecoder, verify_port
    verify_port('models/dec_new.h5')          # run this first, once
    dec = TorchDecoder.from_h5('models/dec_new.h5')
    probs = dec(x)                            # x: (N, 1, 32, 32) float in [0, 1]

Upstream code is AGPL-3.0. This module reads a weights file; it does not copy source.
Cite Berral-Soler et al., Image and Vision Computing 152 (2024) 105313.
"""

import numpy as np

DEFAULT_H5 = "models/dec_new.h5"


# --------------------------------------------------------------------------- weights

def load_weights(h5_path=DEFAULT_H5):
    """Read decoder weights from the Keras .h5. Returns a dict of numpy arrays."""
    import h5py

    store = {}
    with h5py.File(h5_path, "r") as f:
        def visit(name, obj):
            if isinstance(obj, h5py.Dataset) and name.startswith("model_weights/"):
                store[name] = np.array(obj)
        f.visititems(visit)

    def pick(layer, suffix):
        hits = [k for k in store if f"/{layer}/" in k and k.endswith(suffix)]
        if len(hits) != 1:
            raise KeyError(f"expected one match for {layer}:{suffix}, got {hits}")
        return store[hits[0]]

    w = {
        "c1_w": pick("conv2d", "kernel:0"),   "c1_b": pick("conv2d", "bias:0"),
        "p1_a": pick("p_re_lu", "alpha:0"),
        "c2_w": pick("conv2d_1", "kernel:0"), "c2_b": pick("conv2d_1", "bias:0"),
        "p2_a": pick("p_re_lu_1", "alpha:0"),
        "c3_w": pick("conv2d_2", "kernel:0"), "c3_b": pick("conv2d_2", "bias:0"),
    }
    expect = {"c1_w": (5,5,1,8), "c1_b": (8,), "p1_a": (28,28,8),
              "c2_w": (3,3,8,4), "c2_b": (4,), "p2_a": (12,12,4),
              "c3_w": (1,1,4,1), "c3_b": (1,)}
    for k, shape in expect.items():
        assert w[k].shape == shape, f"{k}: got {w[k].shape}, expected {shape}"
    return w


# ------------------------------------------------------------------- numpy reference

class NumpyDecoder:
    """Reference implementation. Verified against synthetic markers: 4/5 exact."""

    def __init__(self, weights):
        self.w = weights

    @classmethod
    def from_h5(cls, path=DEFAULT_H5):
        return cls(load_weights(path))

    @staticmethod
    def _conv(x, w, b):
        kh, kw, _, co = w.shape
        H, W, _ = x.shape
        out = np.zeros((H - kh + 1, W - kw + 1, co), np.float32)
        for i in range(out.shape[0]):
            for j in range(out.shape[1]):
                out[i, j] = np.tensordot(
                    x[i:i+kh, j:j+kw, :], w, axes=([0, 1, 2], [0, 1, 2])) + b
        return out

    @staticmethod
    def _pool(x):
        h, w, c = x.shape
        return x.reshape(h // 2, 2, w // 2, 2, c).max(axis=(1, 3))

    def __call__(self, marker32):
        """marker32: (32, 32) float in [0, 1]. Returns (6, 6) bit probabilities."""
        w = self.w
        x = np.asarray(marker32, np.float32)[..., None]
        x = self._conv(x, w["c1_w"], w["c1_b"])
        x = np.where(x > 0, x, w["p1_a"] * x)
        x = self._pool(x)
        x = self._conv(x, w["c2_w"], w["c2_b"])
        x = np.where(x > 0, x, w["p2_a"] * x)
        x = self._pool(x)
        x = self._conv(x, w["c3_w"], w["c3_b"])
        return (1.0 / (1.0 + np.exp(-np.clip(x[..., 0], -60, 60)))).astype(np.float32)


# -------------------------------------------------------------------------- pytorch

def _torch():
    import torch
    import torch.nn as nn
    return torch, nn


class SpatialPReLU:
    """Created via _build so torch is imported lazily. See module docstring."""


def TorchDecoder_factory():
    torch, nn = _torch()

    class _SpatialPReLU(nn.Module):
        """Per-element PReLU, matching Keras PReLU(shared_axes=None)."""
        def __init__(self, alpha_chw):
            super().__init__()
            self.alpha = nn.Parameter(torch.tensor(alpha_chw, dtype=torch.float32))

        def forward(self, x):
            return torch.where(x > 0, x, self.alpha.unsqueeze(0) * x)

    class _TorchDecoder(nn.Module):
        def __init__(self, weights=None):
            super().__init__()
            self.c1 = nn.Conv2d(1, 8, 5)
            self.c2 = nn.Conv2d(8, 4, 3)
            self.c3 = nn.Conv2d(4, 1, 1)
            self.pool = nn.MaxPool2d(2)
            a1 = np.zeros((8, 28, 28), np.float32) if weights is None \
                 else np.transpose(weights["p1_a"], (2, 0, 1))
            a2 = np.zeros((4, 12, 12), np.float32) if weights is None \
                 else np.transpose(weights["p2_a"], (2, 0, 1))
            self.p1, self.p2 = _SpatialPReLU(a1), _SpatialPReLU(a2)
            if weights is not None:
                self.load_keras(weights)

        def load_keras(self, w):
            with torch.no_grad():
                for conv, kw, kb in ((self.c1, "c1_w", "c1_b"),
                                     (self.c2, "c2_w", "c2_b"),
                                     (self.c3, "c3_w", "c3_b")):
                    conv.weight.copy_(torch.tensor(np.transpose(w[kw], (3, 2, 0, 1))))
                    conv.bias.copy_(torch.tensor(w[kb]))

        @classmethod
        def from_h5(cls, path=DEFAULT_H5):
            return cls(load_weights(path))

        def forward(self, x):
            """x: (N, 1, 32, 32) in [0, 1]. Returns (N, 6, 6) bit probabilities."""
            x = self.pool(self.p1(self.c1(x)))
            x = self.pool(self.p2(self.c2(x)))
            return torch.sigmoid(self.c3(x)).squeeze(1)

    return _TorchDecoder


def __getattr__(name):
    if name == "TorchDecoder":
        return TorchDecoder_factory()
    raise AttributeError(name)


# --------------------------------------------------------------------------- verify

def verify_port(h5_path=DEFAULT_H5, n=16, tol=1e-4, verbose=True):
    """Assert the PyTorch port matches the numpy reference. Run once before use.

    The numpy path is validated; the torch path is validated against it here rather
    than assumed. Do not skip this.
    """
    torch, _ = _torch()
    w = load_weights(h5_path)
    ref = NumpyDecoder(w)
    net = TorchDecoder_factory()(w).eval()

    rng = np.random.default_rng(0)
    x = rng.random((n, 32, 32)).astype(np.float32)
    with torch.no_grad():
        got = net(torch.tensor(x).unsqueeze(1)).numpy()
    want = np.stack([ref(xi) for xi in x])

    err = float(np.abs(got - want).max())
    if verbose:
        print(f"port check: {n} random inputs, max abs difference {err:.2e} (tol {tol:g})")
    assert err < tol, f"PyTorch port disagrees with numpy reference (max diff {err:.2e})"
    if verbose:
        print("PyTorch port matches the numpy reference.")
    return err


def decode_ids(probs, id_to_bits, n_markers=250):
    """Hard-decision decode: nearest codeword by Hamming distance over 4 rotations.

    probs: (6, 6) bit probabilities. Returns (marker_id, rotation, hamming).
    Replace with soft log-likelihood decoding to use the confidences.
    """
    obs = (np.asarray(probs) > 0.5).astype(np.uint8)
    best = (None, None, 99)
    for mid in range(n_markers):
        ref = np.asarray(id_to_bits(mid), np.uint8).reshape(6, 6)
        for rot in range(4):
            h = int(np.sum(np.rot90(obs, rot) != ref))
            if h < best[2]:
                best = (mid, rot, h)
    return best


if __name__ == "__main__":
    import sys
    path = sys.argv[1] if len(sys.argv) > 1 else DEFAULT_H5
    w = load_weights(path)
    print("weights loaded:", {k: v.shape for k, v in w.items()})
    try:
        verify_port(path)
    except ImportError:
        print("torch not installed; numpy reference verified only")


Writing deeparuco_port.py


In [3]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'   # deterministic cuBLAS, must be set before torch
import sys, json, glob, zlib, subprocess, time, itertools
import numpy as np, cv2, torch
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE, WORK = '/content/drive/MyDrive/aruco_p2', '/content/work'
except ImportError:
    BASE = os.environ.get('ARUCO_P2_BASE',
                          os.path.join(os.path.expanduser('~'), 'aruco_p2')).replace('\\', '/')
    WORK = f'{BASE}/work'
ART, CACHE = f'{BASE}/artifacts', f'{BASE}/archives'
os.makedirs(WORK, exist_ok=True); sys.path.insert(0, os.getcwd())
assert os.path.exists(f'{ART}/splits.json'), f'{ART}/splits.json missing, run N0 first'

REPO = f'{WORK}/deeparuco'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--depth','1','https://github.com/AVAuco/deeparuco.git',REPO],check=True)
sys.path.insert(0, REPO)
from impl.aruco import id_to_bits
from deeparuco_port import load_weights, NumpyDecoder

DEV_ = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
splits = json.load(open(f'{ART}/splits.json'))
DEVV, HELDOUT, STRIDE = splits['dev'], splits['heldout'][0], splits['benchmark_stride']
Wk  = load_weights(f'{REPO}/models/dec_new.h5')
CB  = np.stack([np.asarray(id_to_bits(i),np.uint8).reshape(6,6) for i in range(250)])
ROT = np.stack([np.rot90(CB,k,axes=(1,2)).reshape(250,36) for k in range(4)]).astype(np.float32)

SHADOW = f'{WORK}/shadow'; os.makedirs(SHADOW, exist_ok=True)
for i in range(1,7):
    if not os.path.isdir(f'{SHADOW}/video_{i}'):
        tar = f'{CACHE}/video_{i}.tar.gz'
        assert os.path.exists(tar), f'{tar} missing, see the download cell in N0'
        subprocess.run(['tar','-xzf',tar,'-C',SHADOW], check=True)

N0M = json.load(open(f'{ART}/manifest.json'))
for vid in DEVV + [HELDOUT]:
    n = len(glob.glob(f'{SHADOW}/{vid}/*.png'))
    assert n == N0M['shadow_aruco']['frames'][vid], f'{vid}: {n} frames on disk, manifest says other'
print('torch', torch.__version__, '| device', DEV_)
if DEV_.type != 'cuda':
    print('\n*** THIS IS A CPU RUNTIME. One training run takes about 40 minutes here')
    print('*** and about 2 minutes on a GPU. Use Runtime, Change runtime type, T4 or L4,')
    print('*** then rerun every cell from the top before training.')
print('dev:', DEVV, '| held out:', HELDOUT, '| stride', STRIDE)


Mounted at /content/drive
torch 2.14.1+cu130 | device cuda
dev: ['video_1', 'video_2', 'video_3', 'video_4', 'video_5'] | held out: video_6 | stride 6


In [4]:
r_int = lambda v: int(round(v))
VALUES = {'grey':127, 'black':0, 'white':255}

def ordered_corners(xs, ys):
    cx, cy = np.mean(xs), np.mean(ys)
    idx = np.argsort(np.arctan2(xs-cx, ys-cy))
    return [v for i in idx for v in (xs[i], ys[i])]

def warp32(crop, c, t=32):
    dst = np.array([[0,0],[0,t-1],[t-1,t-1],[t-1,0]], np.float32)
    src = np.clip(np.array([[c[0]*63,c[1]*63],[c[2]*63,c[3]*63],
                            [c[4]*63,c[5]*63],[c[6]*63,c[7]*63]]),0,63).astype(np.float32)
    return cv2.warpPerspective(crop, cv2.getPerspectiveTransform(src,dst), (t,t))

def get_corners(m):
    c = m['corners']
    if isinstance(c[0], dict): return np.array([[p['x'],p['y']] for p in c], np.float32)
    return np.asarray(c, np.float32).reshape(4,2)

def cell_occlusion(m32):
    s = 32/8.0; out = np.zeros((6,6), bool)
    for r in range(6):
        for c in range(6):
            out[r,c] = m32[int((r+1)*s):int((r+2)*s), int((c+1)*s):int((c+2)*s)].mean() > 127.5
    return out

def prepare(img, mask, corners, scale=None):
    x, y = corners[:,0], corners[:,1]
    mnx, mxx, mny, mxy = x.min(), x.max(), y.min(), y.max()
    ex, ey = 0.2*(mxx-mnx)+0.5, 0.2*(mxy-mny)+0.5
    mnx, mny = max(mnx-ex,0), max(mny-ey,0)
    mxx = min(mxx+ex, img.shape[1]-1); mxy = min(mxy+ey, img.shape[0]-1)
    if mxx-mnx < 8 or mxy-mny < 8: return None, None
    sl = (slice(r_int(mny), r_int(mxy)), slice(r_int(mnx), r_int(mxx)))
    crop = cv2.resize(img[sl], (64,64)).astype(np.float32)
    lo, hi = (crop.min(), crop.max()) if scale is None else scale
    crop = np.clip((crop - lo)/(hi - lo + 1e-9), 0, 1)
    nc = ordered_corners((x-mnx)/(mxx-mnx), (y-mny)/(mxy-mny))
    m32 = warp32(cv2.resize(mask[sl],(64,64)), nc) if mask.any() else np.zeros((32,32), np.uint8)
    return warp32(crop, nc), cell_occlusion(m32)

def occluder_box(img, corners, level, position, key, mid):
    if level <= 0: return None
    x, y = corners[:,0], corners[:,1]
    x0,x1,y0,y1 = x.min(),x.max(),y.min(),y.max(); w,h = x1-x0, y1-y0
    if position == 'edge':
        side = int(np.random.default_rng(zlib.crc32(f'occlusion|{level}|{key}|{mid}'.encode())).integers(0,4))
        box = [(x0,y0,x0+level*w,y1),(x1-level*w,y0,x1,y1),
               (x0,y0,x1,y0+level*h),(x0,y1-level*h,x1,y1)][side]
    else:
        cx, cy = (x0+x1)/2, (y0+y1)/2
        box = (cx-level*w/2, cy-level*h/2, cx+level*w/2, cy+level*h/2)
    a,b,c,d = [int(round(v)) for v in box]
    a,b = max(a,0),max(b,0); c,d = min(c,img.shape[1]),min(d,img.shape[0])
    return (a,b,c,d) if (c>a and d>b) else None

def apply_occluder(img, box, value):
    if box is None: return img, np.zeros(img.shape[:2], np.uint8)
    if isinstance(value, str): value = VALUES[value]
    a,b,c,d = box
    out = img.copy(); mk = np.zeros(img.shape[:2], np.uint8)
    out[b:d,a:c] = int(value); mk[b:d,a:c] = 255
    return out, mk

def pairs_for(vid):
    out = []
    for f in sorted(glob.glob(f'{SHADOW}/{vid}/*.png')):
        j = f'{SHADOW}/{vid}/corrected_annotations/{os.path.splitext(os.path.basename(f))[0]}.json'
        if os.path.exists(j): out.append((f,j))
    return out

class SpatialPReLU(nn.Module):
    def __init__(s,c,h,w): super().__init__(); s.a = nn.Parameter(torch.zeros(c,h,w))
    def forward(s,x): return torch.where(x>0, x, s.a.unsqueeze(0)*x)

class Decoder(nn.Module):
    """8 and 4 filters, as in the released code. pool is 'max' or 'avg'."""
    def __init__(s, out_ch=1, pool='max'):
        super().__init__()
        s.c1=nn.Conv2d(1,8,5); s.p1=SpatialPReLU(8,28,28)
        s.c2=nn.Conv2d(8,4,3); s.p2=SpatialPReLU(4,12,12)
        s.c3=nn.Conv2d(4,out_ch,1); s.out_ch=out_ch
        s.pool = F.max_pool2d if pool=='max' else F.avg_pool2d
    def forward(s,x):
        x=s.pool(s.p1(s.c1(x)),2); x=s.pool(s.p2(s.c2(x)),2); x=s.c3(x)
        return (x[:,0], x[:,1]) if s.out_ch==2 else (x[:,0], None)
print('pipeline ready')


pipeline ready


In [5]:
def _conv(x,w,b):
    kh,kw,_,_ = w.shape
    return np.einsum('nohckl,klci->nohi', sliding_window_view(x,(kh,kw),axis=(1,2)), w, optimize=True)+b
def _pool(x):
    N,H,Wd,Cc = x.shape; return x.reshape(N,H//2,2,Wd//2,2,Cc).max(axis=(2,4))

def decode_published(x8, chunk=4096):
    out=[]
    for s in range(0,len(x8),chunk):
        x = x8[s:s+chunk].astype(np.float32)[...,None]/255.0
        x = _pool(np.where((y:=_conv(x,Wk['c1_w'],Wk['c1_b']))>0,y,Wk['p1_a']*y))
        x = _pool(np.where((y:=_conv(x,Wk['c2_w'],Wk['c2_b']))>0,y,Wk['p2_a']*y))
        out.append(1/(1+np.exp(-np.clip(_conv(x,Wk['c3_w'],Wk['c3_b'])[...,0],-60,60))))
    return np.concatenate(out).astype(np.float32)

def decode_torch(net, x8, chunk=8192):
    ps, es = [], []
    with torch.no_grad():
        for s in range(0,len(x8),chunk):
            xb = torch.tensor(x8[s:s+chunk], dtype=torch.float32).unsqueeze(1).to(DEV_)/255.0
            lo, el = net(xb)
            ps.append(torch.sigmoid(lo).cpu().numpy())
            es.append(torch.sigmoid(el).cpu().numpy() if el is not None else np.zeros((len(lo),6,6),np.float32))
    return np.concatenate(ps), np.concatenate(es)

def dec_soft(P, Wt=None):
    LP = np.log(np.clip(P,1e-6,1)).reshape(len(P),36)
    LQ = np.log(np.clip(1-P,1e-6,1)).reshape(len(P),36)
    if Wt is not None:
        w = Wt.reshape(len(P),36); LP, LQ = LP*w, LQ*w
    return np.stack([LP@ROT[k].T + LQ@(1-ROT[k]).T for k in range(4)]).max(0).argmax(1)
def hamming_w(P, W=None):
    B = (P.reshape(len(P),36) > 0.5).astype(np.float32)
    W = np.ones_like(B) if W is None else W.reshape(len(P),36).astype(np.float32)
    return np.stack([(B*W)@(1-ROT[k]).T + ((1-B)*W)@ROT[k].T for k in range(4)]).min(0)

def hard_expected(P, ids, W=None):
    """Exact expected accuracy with fair random tie breaking, per marker (0 to 1)."""
    D = hamming_w(P, W); tied = D == D.min(1, keepdims=True)
    return tied[np.arange(len(ids)), ids] / tied.sum(1)
print('decoders ready')


decoders ready


In [6]:
def paired(a_ok, b_ok, groups, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    uf, inv = np.unique(groups, return_inverse=True)
    by = [np.where(inv==i)[0] for i in range(len(uf))]
    obs = a_ok.mean() - b_ok.mean()
    d = np.empty(n_boot)
    for i in range(n_boot):
        idx = np.concatenate([by[p] for p in rng.integers(0, len(by), len(by))])
        d[i] = a_ok[idx].mean() - b_ok[idx].mean()
    lo, hi = np.percentile(d, [2.5, 97.5])
    p = 2*min((d <= 0).mean(), (d >= 0).mean())
    return float(obs), float(lo), float(hi), max(float(p), 1/n_boot)
from scipy import stats as sst

def welch(a, b):
    """Unpaired difference of means across seeds with a Welch t interval."""
    a, b = np.asarray(a, float), np.asarray(b, float); m = a.mean() - b.mean()
    va, vb = a.var(ddof=1)/len(a), b.var(ddof=1)/len(b); se = np.sqrt(va + vb)
    if se == 0: return float(m), float(m), float(m)
    df = (va + vb)**2 / (va**2/(len(a)-1) + vb**2/(len(b)-1))
    h = sst.t.ppf(0.975, df) * se
    return float(m), float(m - h), float(m + h)
print('statistics ready')


statistics ready


In [7]:
# ---------------------------------------------------------------- Flying ArUco v2
FLY = f'{WORK}/flyingarucov2'
if not os.path.isdir(FLY):
    os.makedirs(FLY, exist_ok=True)
    tar = f'{CACHE}/flyingarucov2.tar.gz'
    assert os.path.exists(tar), f'{tar} missing, see N0'
    subprocess.run(['tar','-xzf',tar,'-C',FLY], check=True)
fly_json = sorted(glob.glob(f'{FLY}/**/*.json', recursive=True))
assert len(fly_json) == N0M['flying_aruco_v2']['images'], \
    f'{len(fly_json)} annotations on disk, N0 manifest says {N0M["flying_aruco_v2"]["images"]}'

def bits_for(mid, rot, conv):
    b = CB[mid]; k = rot if conv in (0,1) else -rot
    return np.rot90(b, k) if conv in (0,2) else np.rot90(b, -k)

ref = NumpyDecoder(Wk); probe = []
for jp in fly_json[:60]:
    img = cv2.imread(jp.replace('.json','.jpg'), cv2.IMREAD_GRAYSCALE)
    if img is None: continue
    for m in json.load(open(jp))['markers'][:4]:
        x32, _ = prepare(img, np.zeros(img.shape[:2], np.uint8), get_corners(m))
        if x32 is None: continue
        probe.append((ref(x32), int(m['id']), int(m.get('rot', m.get('rotation', 0)))))
assert len(probe) > 20, f'only {len(probe)} probe markers, Flying ArUco did not load'
scores = {c: float(np.mean([np.mean((p>0.5).astype(np.uint8)==bits_for(i,r,c)) for p,i,r in probe]))
          for c in range(4)}
CONV = max(scores, key=scores.get)
assert scores[CONV] > 0.90, f'no rotation convention agrees ({scores}), labels would be wrong'
print(f'{len(fly_json)} Flying ArUco v2 annotations | rotation convention {CONV} ({scores[CONV]:.4f})')

2500 Flying ArUco v2 annotations | rotation convention 0 (0.9912)


In [8]:
# ---------------------------------------------------------------- training crops
OCC_TRAIN = [0.0, 0.10, 0.20, 0.25, 0.35, 0.45, 0.55]
N_SAMPLES = 5

def build_training_set(value):
    """value is 127 for arms B, C, D, M, G, 'random' for E and F, 0 for H."""
    X, Ybit, Yocc = [], [], []
    t0 = time.time()
    for n, jp in enumerate(fly_json):
        img = cv2.imread(jp.replace('.json','.jpg'), cv2.IMREAD_GRAYSCALE)
        if img is None: continue
        key = os.path.basename(jp)
        for m in json.load(open(jp))['markers']:
            cn, mid = get_corners(m), int(m['id'])
            if mid >= 250: continue
            bits = bits_for(mid, int(m.get('rot', m.get('rotation', 0))), CONV)
            rng = np.random.default_rng(zlib.crc32(f'train|{key}|{mid}'.encode()))
            for L in rng.choice(OCC_TRAIN, N_SAMPLES, replace=False):
                v = int(rng.integers(0,256)) if value == 'random' else value
                box = occluder_box(img, cn, float(L), 'edge', key, mid)
                oi, om = apply_occluder(img, box, v)
                x32, occ = prepare(oi, om, cn)
                if x32 is None: continue
                X.append((x32*255).astype(np.uint8)); Ybit.append(bits); Yocc.append(occ)
        if n % 800 == 0: print(f'  {value}: {n}/{len(fly_json)} ({time.time()-t0:.0f}s) crops={len(X)}')
    X = np.stack(X); Ybit = np.stack(Ybit).astype(np.float32); Yocc = np.stack(Yocc).astype(np.float32)
    assert len(X) > 50000, f'only {len(X)} crops, expected about 64840'
    print(f'  {value}: {len(X)} crops, mean occluded cells {Yocc.sum((1,2)).mean():.2f}, '
          f'{time.time()-t0:.0f}s')
    return X, Ybit, Yocc

SETS = {127: build_training_set(127), 'random': build_training_set('random'), 0: build_training_set(0)}
assert len({len(v[0]) for v in SETS.values()}) == 1, 'training sets differ in size, pairing would break'
print('\ntraining sets ready')

  127: 0/2500 (0s) crops=40
  127: 800/2500 (18s) crops=20830
  127: 1600/2500 (36s) crops=41645
  127: 2400/2500 (54s) crops=62275
  127: 64840 crops, mean occluded cells 7.00, 58s
  random: 0/2500 (0s) crops=40
  random: 800/2500 (19s) crops=20830
  random: 1600/2500 (37s) crops=41645
  random: 2400/2500 (55s) crops=62275
  random: 64840 crops, mean occluded cells 7.00, 58s
  0: 0/2500 (0s) crops=40
  0: 800/2500 (19s) crops=20830
  0: 1600/2500 (37s) crops=41645
  0: 2400/2500 (55s) crops=62275
  0: 64840 crops, mean occluded cells 7.00, 57s

training sets ready


In [9]:
# =================================================================================
# Everything the training loop needs is defined in this one cell, so a runtime
# restart cannot leave half of it missing.
# =================================================================================
CKPT = f'{BASE}/ckpt_n14'; os.makedirs(CKPT, exist_ok=True)   # reuse N14 runs; only H is new
CODE_TAG = 'n14-v1'         # change this whenever the training code changes

SEEDS   = [0, 1, 2, 3, 4]
MARGIN  = 1.0               # equivalence margin, in accuracy points
EPOCHS, PATIENCE, BS = 200, 20, 256
ALLOW_CPU = False           # set True only if you accept about 40 minutes per run
ALLOW_NONDETERMINISM = False

# arm -> output channels, loss masking, pooling, training occluder brightness
ARMS = {
 'B': dict(out_ch=1, mask_bits=False, pool='max', value=127),
 'C': dict(out_ch=2, mask_bits=True,  pool='max', value=127),
 'D': dict(out_ch=2, mask_bits=False, pool='max', value=127),
 'E': dict(out_ch=1, mask_bits=False, pool='max', value='random'),
 'F': dict(out_ch=2, mask_bits=True,  pool='max', value='random'),
 'H': dict(out_ch=2, mask_bits=True,  pool='max', value=0),
 'M': dict(out_ch=1, mask_bits=False, pool='max', value=127),
 'G': dict(out_ch=1, mask_bits=False, pool='avg', value=127),
}
assert DEV_.type == 'cuda' or ALLOW_CPU, (
    'This is a CPU runtime and training would take many hours. '
    'Switch to a GPU (Runtime, Change runtime type), rerun every cell from the top, '
    'or set ALLOW_CPU = True if you really mean it.')

GPU_SETS = {}
for v, (X, Yb, Yo) in SETS.items():
    GPU_SETS[v] = (torch.from_numpy(X).to(DEV_),                 # uint8 (N,32,32)
                   torch.from_numpy(Yb).to(DEV_),                # float32 (N,6,6)
                   torch.from_numpy(Yo).to(DEV_))
    print(f'  set {v}: {X.nbytes/1e6:.0f} MB of crops moved to {DEV_}')

SPLIT = {}
for v, (X, _, _) in SETS.items():
    idx = np.random.default_rng(42).permutation(len(X)); cut = int(0.9*len(X))
    SPLIT[v] = (torch.from_numpy(idx[:cut]).to(DEV_), torch.from_numpy(idx[cut:]).to(DEV_))

def signature(arm):
    return json.dumps(dict(ARMS[arm], code=CODE_TAG, epochs=EPOCHS, patience=PATIENCE, bs=BS),
                      sort_keys=True, default=str)

def train_arm(arm, seed, epochs=EPOCHS, patience=PATIENCE, save=True):
    cfg = ARMS[arm]
    Xg, Bg, Og = GPU_SETS[cfg['value']]
    tr, va = SPLIT[cfg['value']]
    path = f'{CKPT}/arm_{arm}_s{seed}.pt'
    if save and os.path.exists(path):
        ck = torch.load(path, map_location=DEV_)
        assert isinstance(ck, dict) and ck.get('sig') == signature(arm), \
            f'{path} was made with other settings or code. Delete it and rerun.'
        net = Decoder(cfg['out_ch'], cfg['pool']).to(DEV_)
        net.load_state_dict(ck['state']); net.eval()
        return net, None, 0.0
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    rng = np.random.default_rng(seed)
    gen = torch.Generator(device=DEV_); gen.manual_seed(seed)
    net = Decoder(cfg['out_ch'], cfg['pool']).to(DEV_)
    opt = torch.optim.Adam(net.parameters(), 1e-3)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=8)

    def run(ids, train):
        net.train(train)
        order = ids[torch.randperm(len(ids), device=DEV_, generator=gen)] if train else ids
        tot = torch.zeros((), device=DEV_); n = 0
        with torch.set_grad_enabled(train):
            for s in range(0, len(order), BS):
                b  = order[s:s+BS]
                xb = Xg[b].unsqueeze(1).float()/255.0
                bb = Bg[b]; ob = Og[b]
                if train:                                    # same augmentation as N3b
                    k, fl = int(rng.integers(0,4)), int(rng.integers(0,3))
                    if k:
                        xb = torch.rot90(xb,k,dims=(-2,-1))
                        bb = torch.rot90(bb,k,dims=(-2,-1)); ob = torch.rot90(ob,k,dims=(-2,-1))
                    if fl == 1:
                        xb, bb, ob = torch.flip(xb,(-1,)), torch.flip(bb,(-1,)), torch.flip(ob,(-1,))
                    elif fl == 2:
                        xb, bb, ob = torch.flip(xb,(-2,)), torch.flip(bb,(-2,)), torch.flip(ob,(-2,))
                logit, elogit = net(xb)
                if cfg['mask_bits']:
                    vis  = 1.0 - ob
                    loss = (F.binary_cross_entropy_with_logits(logit, bb, reduction='none')*vis
                            ).sum()/vis.sum().clamp(min=1)
                else:
                    loss = F.binary_cross_entropy_with_logits(logit, bb)
                if cfg['out_ch'] == 2:
                    loss = loss + F.binary_cross_entropy_with_logits(elogit, ob)
                if train:
                    opt.zero_grad(); loss.backward(); opt.step()
                tot = tot + loss.detach()*len(b); n += len(b)
        return (tot/n).item()            # one sync per epoch, not one per batch

    best, bad, t0, state, ep = 1e9, 0, time.time(), None, 0
    for ep in range(epochs):
        run(tr, True); vl = run(va, False); sch.step(vl)
        if vl < best - 1e-5:
            best, bad = vl, 0
            state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(state); net.eval()
    if save:                                             # written only once the run is complete
        tmp = path + '.tmp'
        torch.save(dict(state=state, sig=signature(arm), best=best, epochs=ep+1), tmp)
        os.replace(tmp, path)
    return net, best, time.time()-t0

# ----- determinism probe: the same short run twice must give identical weights
netA, _, secs = train_arm('B', SEEDS[0], epochs=3, patience=99, save=False)
netB, _, _    = train_arm('B', SEEDS[0], epochs=3, patience=99, save=False)
diff = max(float((a-b).abs().max()) for a, b in zip(netA.state_dict().values(), netB.state_dict().values()))
print(f'\ndeterminism probe: same seed twice, largest weight difference {diff:.2e}')
assert diff == 0.0 or ALLOW_NONDETERMINISM, (
    'Training is still not deterministic. Restart the session (Runtime, Restart session) and '
    'Run all, so the setting is applied before torch starts. If it still fails, set '
    'ALLOW_NONDETERMINISM = True and report the B against M gap as run to run noise.')
per_epoch = secs/3
print(f'probe: 3 epochs of arm B took {secs:.1f}s, so {per_epoch:.2f}s per epoch')
print(f'projected: about {per_epoch*150/60:.1f} min per run, '
      f'{per_epoch*150*len(ARMS)*len(SEEDS)/3600:.1f} hours for all {len(ARMS)*len(SEEDS)} runs')
assert per_epoch < 5, 'still too slow, stop and check the device before continuing'


  set 127: 66 MB of crops moved to cuda
  set random: 66 MB of crops moved to cuda
  set 0: 66 MB of crops moved to cuda

determinism probe: same seed twice, largest weight difference 0.00e+00
probe: 3 epochs of arm B took 4.0s, so 1.35s per epoch
projected: about 3.4 min per run, 2.2 hours for all 40 runs


In [10]:
MODELS, TIMES = {}, {}
total = len(ARMS)*len(SEEDS); k = 0
t_all = time.time()
for arm in ARMS:
    for s in SEEDS:
        k += 1
        net, vl, secs = train_arm(arm, s)
        MODELS[(arm,s)] = net; TIMES[(arm,s)] = secs
        tag = 'loaded from checkpoint' if vl is None else f'val {vl:.4f} ({secs:.0f}s)'
        print(f'  [{k:>2}/{total}] arm {arm} seed {s}: {tag}', flush=True)
print(f'\nall {total} runs done in {(time.time()-t_all)/60:.1f} min')

  [ 1/40] arm B seed 0: loaded from checkpoint
  [ 2/40] arm B seed 1: loaded from checkpoint
  [ 3/40] arm B seed 2: loaded from checkpoint
  [ 4/40] arm B seed 3: loaded from checkpoint
  [ 5/40] arm B seed 4: loaded from checkpoint
  [ 6/40] arm C seed 0: loaded from checkpoint
  [ 7/40] arm C seed 1: loaded from checkpoint
  [ 8/40] arm C seed 2: loaded from checkpoint
  [ 9/40] arm C seed 3: loaded from checkpoint
  [10/40] arm C seed 4: loaded from checkpoint
  [11/40] arm D seed 0: loaded from checkpoint
  [12/40] arm D seed 1: loaded from checkpoint
  [13/40] arm D seed 2: loaded from checkpoint
  [14/40] arm D seed 3: loaded from checkpoint
  [15/40] arm D seed 4: loaded from checkpoint
  [16/40] arm E seed 0: loaded from checkpoint
  [17/40] arm E seed 1: loaded from checkpoint
  [18/40] arm E seed 2: loaded from checkpoint
  [19/40] arm E seed 3: loaded from checkpoint
  [20/40] arm E seed 4: loaded from checkpoint
  [21/40] arm F seed 0: loaded from checkpoint
  [22/40] arm

In [11]:
TARGETS  = [3, 6, 9, 12, 15, 18]
COLOURS  = ['grey', 'white', 'black']
POSITION = 'edge'

cal = []
for vid in DEVV[:2]:
    for f, j in pairs_for(vid)[::STRIDE*8]:
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        key = f'{vid}/{os.path.splitext(os.path.basename(f))[0]}'
        for m in json.load(open(j))['markers']:
            cal.append((img, get_corners(m), key, int(m['id'])))
        if len(cal) > 400: break
    if len(cal) > 400: break
def mean_cells(level):
    tot=n=0
    for img, cn, key, mid in cal:
        oi, om = apply_occluder(img, occluder_box(img, cn, level, POSITION, key, mid), 'grey')
        _, occ = prepare(oi, om, cn)
        if occ is None: continue
        tot += occ.sum(); n += 1
    return tot/max(n,1)
curve = [(float(L), mean_cells(float(L))) for L in np.arange(0.02, 0.70, 0.02)]
LEVEL_FOR = {T: min(curve, key=lambda lc: abs(lc[1]-T))[0] for T in TARGETS}
print('severity calibration:', {T: round(LEVEL_FOR[T],2) for T in TARGETS})

CLEAN_KEY = ('clean', 0)
CONDS = [(c,T) for c in COLOURS for T in TARGETS]
EV = {k: {'x':[], 'id':[], 'frame':[], 'occ':[], 'mask':[]} for k in [CLEAN_KEY] + CONDS}
def _add(k, x32, occ, mid, key):
    d = EV[k]
    d['x'].append((x32*255).astype(np.uint8)); d['id'].append(mid)
    d['frame'].append(key); d['occ'].append(int(occ.sum())); d['mask'].append(occ)

t0 = time.time()
for vid in DEVV:
    for f, j in pairs_for(vid)[::STRIDE]:
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        key = f'{vid}/{os.path.splitext(os.path.basename(f))[0]}'
        zero = np.zeros(img.shape[:2], np.uint8)
        for m in json.load(open(j))['markers']:
            cn, mid = get_corners(m), int(m['id'])
            xc, oc = prepare(img, zero, cn)
            if xc is None: continue                      # same corners, so every condition would fail too
            _add(CLEAN_KEY, xc, oc, mid, key)
            for T in TARGETS:
                box = occluder_box(img, cn, LEVEL_FOR[T], POSITION, key, mid)
                for col in COLOURS:
                    oi, om = apply_occluder(img, box, col)
                    x32, occ = prepare(oi, om, cn)
                    _add((col,T), x32, occ, mid, key)
    print(f'  {vid} ({time.time()-t0:.0f}s)')
for k in EV: EV[k] = {kk: np.array(vv) for kk, vv in EV[k].items()}
n = len(EV[CLEAN_KEY]['id'])
assert all(len(EV[k]['id']) == n for k in EV), 'conditions are misaligned'
print(f'\neval set: {n} markers per condition, plus a clean condition')
assert n > 1500, f'only {n} markers, data did not load correctly'


severity calibration: {3: 0.24, 6: 0.3, 9: 0.34, 12: 0.4, 15: 0.46, 18: 0.5}
  video_1 (49s)
  video_2 (87s)
  video_3 (133s)
  video_4 (179s)
  video_5 (215s)

eval set: 12276 markers per condition, plus a clean condition


In [12]:
# Centre occluders: a square in the middle of the marker. The structured decoder assumes a
# strip from one side, so this is where its assumption is wrong.
C_TARGETS = [6, 12, 18]
def mean_cells_at(level, position):
    tot = n = 0
    for img, cn, key, mid in cal:
        oi, om = apply_occluder(img, occluder_box(img, cn, level, position, key, mid), 'grey')
        _, occ = prepare(oi, om, cn)
        if occ is None: continue
        tot += occ.sum(); n += 1
    return tot/max(n, 1)
curve_c = [(float(L), mean_cells_at(float(L), 'centre')) for L in np.arange(0.04, 1.0, 0.04)]
LEVEL_C = {T: min(curve_c, key=lambda lc: abs(lc[1]-T))[0] for T in C_TARGETS}
print('centre severity calibration:', {T: round(LEVEL_C[T], 2) for T in C_TARGETS})

CCONDS = [('c_'+c, T) for c in COLOURS for T in C_TARGETS]
for k in CCONDS: EV[k] = {'x':[], 'id':[], 'frame':[], 'occ':[], 'mask':[]}
t0 = time.time()
for vid in DEVV:
    for f, j in pairs_for(vid)[::STRIDE]:
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        key = f'{vid}/{os.path.splitext(os.path.basename(f))[0]}'
        zero = np.zeros(img.shape[:2], np.uint8)
        for m in json.load(open(j))['markers']:
            cn, mid = get_corners(m), int(m['id'])
            xc, _ = prepare(img, zero, cn)
            if xc is None: continue                      # same skip rule as the edge set
            for T in C_TARGETS:
                box = occluder_box(img, cn, LEVEL_C[T], 'centre', key, mid)
                for col in COLOURS:
                    oi, om = apply_occluder(img, box, col)
                    x32, occ = prepare(oi, om, cn)
                    d = EV[('c_'+col, T)]
                    d['x'].append((x32*255).astype(np.uint8)); d['id'].append(mid)
                    d['frame'].append(key); d['occ'].append(int(occ.sum())); d['mask'].append(occ)
    print(f'  {vid} ({time.time()-t0:.0f}s)')
for k in CCONDS:
    EV[k] = {kk: np.array(vv) for kk, vv in EV[k].items()}
    assert np.array_equal(EV[k]['frame'], EV[CLEAN_KEY]['frame']) and np.array_equal(EV[k]['id'], EV[CLEAN_KEY]['id'])
print('centre set aligned with the edge set:', {k: round(float(EV[k]['occ'].mean()), 1) for k in CCONDS})


centre severity calibration: {6: 0.28, 12: 0.36, 18: 0.44}
  video_1 (27s)
  video_2 (46s)
  video_3 (73s)
  video_4 (97s)
  video_5 (117s)
centre set aligned with the edge set: {('c_grey', 6): 6.7, ('c_grey', 12): 11.9, ('c_grey', 18): 18.9, ('c_white', 6): 6.7, ('c_white', 12): 11.9, ('c_white', 18): 18.9, ('c_black', 6): 6.7, ('c_black', 12): 11.9, ('c_black', 18): 18.9}


In [13]:
ACC = {}            # (arm, seed, colour, target) -> per marker correctness
for (arm, s), net in MODELS.items():
    for col in COLOURS:
        for T in TARGETS:
            d = EV[(col,T)]
            P, _ = decode_torch(net, d['x'])
            ACC[(arm,s,col,T)] = (dec_soft(P) == d['id'])
print('evaluated', len(ACC), 'arm and condition combinations\n')

def summary(arm, col, T):
    v = np.array([ACC[(arm,s,col,T)].mean() for s in SEEDS])*100
    return v.mean(), v.std(ddof=1)

print('ARM ACCURACY, mean and sd over', len(SEEDS), 'seeds, development data\n')
for col in COLOURS:
    print(f'--- occluder brightness: {col}')
    print(f"{'cells':>7}" + ''.join(f'{a:>16}' for a in ARMS))
    for T in TARGETS:
        line = f'{EV[(col,T)]["occ"].mean():>7.1f}'
        for arm in ARMS:
            m, sd = summary(arm, col, T); line += f'{m:>10.2f}+-{sd:<4.2f}'
        print(line)
    print()

evaluated 720 arm and condition combinations

ARM ACCURACY, mean and sd over 5 seeds, development data

--- occluder brightness: grey
  cells               B               C               D               E               F               H               M               G
    3.2     97.21+-2.19     88.64+-4.53     95.36+-2.00     97.47+-1.54     86.80+-1.44     64.73+-1.47     97.21+-2.19     98.84+-0.25
    5.7     94.87+-2.96     78.96+-7.21     91.76+-2.25     95.88+-1.69     74.56+-3.62     59.68+-1.82     94.87+-2.96     96.50+-1.12
    8.5     92.56+-3.49     72.65+-8.57     89.81+-3.01     93.82+-2.19     66.06+-5.28     55.54+-1.83     92.56+-3.49     94.80+-0.86
   11.5     90.23+-3.61     63.30+-8.98     86.84+-2.69     91.51+-2.88     55.01+-5.74     49.08+-1.25     90.23+-3.61     92.18+-1.69
   15.6     85.82+-3.35     52.82+-10.35     82.09+-3.23     87.38+-2.53     41.83+-7.12     41.26+-1.94     85.82+-3.35     87.54+-1.86
   18.0     80.11+-3.71     41.28+-9.99     76.05

## Determinism check

B and M have identical settings and seeds. With deterministic training their weights must be
identical. If they are, every number below is reproducible from the seed alone.


In [14]:
worst_w = worst_a = 0.0
for s in SEEDS:
    sb, sm = MODELS[('B',s)].state_dict(), MODELS[('M',s)].state_dict()
    dw = max(float((sb[k]-sm[k]).abs().max()) for k in sb)
    da = max(abs(100*(ACC[('B',s,c,T)].mean() - ACC[('M',s,c,T)].mean())) for c in COLOURS for T in TARGETS)
    worst_w, worst_a = max(worst_w, dw), max(worst_a, da)
    print(f'  seed {s}: largest weight difference {dw:.2e}, largest accuracy gap {da:.2f} points')
print()
if worst_w == 0.0:
    print('Deterministic: B and M are identical. Results are reproducible from the seed.')
else:
    print(f'WARNING: B and M differ (weights {worst_w:.2e}, accuracy {worst_a:.2f} points). '
          'Training is not fully deterministic, so treat this gap as run to run noise.')


  seed 0: largest weight difference 0.00e+00, largest accuracy gap 0.00 points
  seed 1: largest weight difference 0.00e+00, largest accuracy gap 0.00 points
  seed 2: largest weight difference 0.00e+00, largest accuracy gap 0.00 points
  seed 3: largest weight difference 0.00e+00, largest accuracy gap 0.00 points
  seed 4: largest weight difference 0.00e+00, largest accuracy gap 0.00 points

Deterministic: B and M are identical. Results are reproducible from the seed.


## Unified evaluation

Every arm plus the published decoder (P), on the clean condition and on grey, white and black:

* `soft`: soft log likelihood
* `hard`: nearest codeword with exact fair tie breaking
* `soft_erasure`: soft decoding weighted by 1 minus the erasure output (C, D and F)
* `soft_oracle`, `hard_oracle`: hidden cells removed using the true mask, an upper bound

Then the grey minus black gap, and how far each arm's best practical decoder falls short of
its own oracle.


In [15]:
DECODERS = ['soft', 'hard', 'soft_erasure', 'soft_oracle', 'hard_oracle']

def score(P, E, d):
    ids = d['id']; vis = 1.0 - d['mask'].astype(np.float32)
    out = {'soft':        (dec_soft(P) == ids).astype(np.float32),
           'hard':        hard_expected(P, ids),
           'soft_oracle': (dec_soft(P, vis) == ids).astype(np.float32),
           'hard_oracle': hard_expected(P, ids, vis)}
    if E is not None:
        out['soft_erasure'] = (dec_soft(P, 1 - E) == ids).astype(np.float32)
    return out

R = {}; t0 = time.time()
for k, d in [(k, EV[k]) for k in [CLEAN_KEY] + CONDS]:
    for dec, v in score(decode_published(d['x']), None, d).items():
        R[('P','pub') + k + (dec,)] = v
for (arm, s), net in MODELS.items():
    for k, d in [(k, EV[k]) for k in [CLEAN_KEY] + CONDS]:
        P, E = decode_torch(net, d['x'])
        for dec, v in score(P, E if ARMS[arm]['out_ch'] == 2 else None, d).items():
            R[(arm, s) + k + (dec,)] = v
print(f'scored {len(R)} combinations in {time.time()-t0:.0f}s\n')

ARM_SEEDS = {'P': ['pub'], **{a: SEEDS for a in ARMS}}
def seed_values(arm, col, T, dec):
    return [100*R[(arm,s,col,T,dec)].mean() for s in ARM_SEEDS[arm] if (arm,s,col,T,dec) in R]
def stat(arm, col, T, dec):
    v = seed_values(arm, col, T, dec)
    if not v: return None
    return float(np.mean(v)), (float(np.std(v, ddof=1)) if len(v) > 1 else 0.0)
def practical(arm):
    return 'soft_erasure' if arm in ARMS and ARMS[arm]['out_ch'] == 2 else 'soft'

print('CLEAN, no occluder (domain shift check)\n')
for arm in ARM_SEEDS:
    a, b = stat(arm,'clean',0,'soft'), stat(arm,'clean',0,'hard')
    line = f'{arm:>4}   soft {a[0]:6.2f}+-{a[1]:<4.2f}   hard {b[0]:6.2f}+-{b[1]:<4.2f}'
    if practical(arm) == 'soft_erasure':
        e = stat(arm,'clean',0,'soft_erasure'); line += f'   soft_erasure {e[0]:6.2f}+-{e[1]:<4.2f}'
    print(line)

print('\nACCURACY BY ARM AND DECODER, mean +- sd over seeds (P is the published decoder, one run)\n')
for col in COLOURS:
    print(f'--- {col}')
    print(f"{'arm':>4}{'decoder':>14}" + ''.join(f'{EV[(col,T)]["occ"].mean():>13.1f}' for T in TARGETS))
    for arm in ARM_SEEDS:
        for dec in DECODERS:
            if stat(arm, col, TARGETS[0], dec) is None: continue
            print(f'{arm:>4}{dec:>14}' + ''.join(
                f'{m:>8.2f}+-{sd:<3.1f}' for m, sd in (stat(arm, col, T, dec) for T in TARGETS)))
        print()

print('GREY MINUS BLACK, best practical decoder per arm (positive means black is harder)\n')
for arm in ARM_SEEDS:
    dec = practical(arm)
    print(f'{arm:>4}{dec:>14}' + ''.join(f'{stat(arm,"grey",T,dec)[0]-stat(arm,"black",T,dec)[0]:>9.2f}'
                                         for T in TARGETS))

print('\nGAP TO ORACLE: soft_oracle minus best practical decoder (small means the arm already')
print('knows which cells are hidden)\n')
for col in COLOURS:
    print(f'--- {col}')
    for arm in ARM_SEEDS:
        dec = practical(arm)
        print(f'{arm:>4}{dec:>14}' + ''.join(
            f'{stat(arm,col,T,"soft_oracle")[0]-stat(arm,col,T,dec)[0]:>9.2f}' for T in TARGETS))
    print()

out = {f'{a}|{s}|{c}|{T}|{dec}': float(v.mean()) for (a,s,c,T,dec), v in R.items()}
with open(f'{ART}/n15_unified_dev.json','w') as fh: json.dump(out, fh, indent=1)
print('saved', f'{ART}/n14_unified_dev.json')


scored 3496 combinations in 500s

CLEAN, no occluder (domain shift check)

   P   soft  99.54+-0.00   hard  99.48+-0.00
   B   soft  99.60+-0.59   hard  98.96+-1.33
   C   soft  99.75+-0.09   hard  99.17+-0.48   soft_erasure  99.74+-0.09
   D   soft  99.25+-0.50   hard  98.07+-1.27   soft_erasure  99.19+-0.49
   E   soft  99.82+-0.15   hard  99.31+-0.62
   F   soft  99.87+-0.06   hard  99.40+-0.32   soft_erasure  99.84+-0.09
   H   soft  98.62+-0.60   hard  98.23+-1.14   soft_erasure  98.58+-0.58
   M   soft  99.60+-0.59   hard  98.96+-1.33
   G   soft  99.86+-0.09   hard  99.49+-0.22

ACCURACY BY ARM AND DECODER, mean +- sd over seeds (P is the published decoder, one run)

--- grey
 arm       decoder          3.2          5.7          8.5         11.5         15.6         18.0
   P          soft   85.70+-0.0   76.33+-0.0   65.40+-0.0   49.22+-0.0   31.97+-0.0   22.59+-0.0
   P          hard   89.03+-0.0   79.43+-0.0   69.45+-0.0   52.54+-0.0   34.67+-0.0   23.35+-0.0
   P   soft_oracl

## Can arm H detect black occluders?

For hidden and visible cells: bit accuracy, mean confidence, share wrong while over 90% sure,
mean negative log probability of the true bit, and for erasure arms `flag`, the share of
cells marked hidden (above 0.5). On hidden cells `flag` is the detection rate, on visible
cells it is the false alarm rate. If H still flags few hidden black cells, black occlusion
cannot be detected from the cells alone.


In [16]:
import warnings; warnings.filterwarnings('ignore', message='Mean of empty slice')
# true bits per marker in the decoders' frame, recovered from the clean crops
Pc = decode_published(EV[CLEAN_KEY]['x']).reshape(-1, 36)
ids_c = EV[CLEAN_KEY]['id']
LPc, LQc = np.log(np.clip(Pc,1e-6,1)), np.log(np.clip(1-Pc,1e-6,1))
rot_score = np.stack([(LPc*ROT[k][ids_c]).sum(1) + (LQc*(1-ROT[k][ids_c])).sum(1) for k in range(4)])
TRUTH = ROT[rot_score.argmax(0), ids_c] > 0.5
assert ((Pc>0.5)==TRUTH).mean() > 0.97, 'orientation recovery failed'

def probs(arm, s, k):
    if arm == 'P': return decode_published(EV[k]['x']).reshape(-1, 36), None
    P, E = decode_torch(MODELS[(arm,s)], EV[k]['x'])
    return P.reshape(-1, 36), (E.reshape(-1, 36) if ARMS[arm]['out_ch'] == 2 else None)

CAL_ARMS = ['P', 'B', 'C', 'F', 'H']
CAL_ROWS = []
for col in COLOURS:
    for T in [6, 12, 18]:
        k = (col,T); M = EV[k]['mask'].reshape(-1, 36)
        for arm in CAL_ARMS:
            for s in ARM_SEEDS[arm]:
                P, E = probs(arm, s, k)
                correct = (P > 0.5) == TRUTH; conf = np.maximum(P, 1-P)
                p_true = np.where(TRUTH, P, 1-P)
                for region, sel in (('hidden', M), ('visible', ~M)):
                    CAL_ROWS.append(dict(arm=arm, seed=s, colour=col, target=T, region=region,
                        bit_acc=float(correct[sel].mean()), conf=float(conf[sel].mean()),
                        conf_wrong=float(((~correct) & (conf > 0.9))[sel].mean()),
                        nll=float(-np.log(np.clip(p_true[sel],1e-6,1)).mean()),
                        flag=float((E[sel] > 0.5).mean()) if E is not None else float('nan')))

def agg(arm, col, T, region, key):
    return np.nanmean([r[key] for r in CAL_ROWS if r['arm']==arm and r['colour']==col
                       and r['target']==T and r['region']==region])

hdr = f"{'acc':>11}{'conf':>7}{'wrong>.9':>9}{'-log p':>8}{'flag':>7}"
print(f"{'':>18} | {'HIDDEN CELLS':^42} | {'VISIBLE CELLS':^42}")
print(f"{'col':>6}{'cells':>7}{'arm':>5} | {hdr} | {hdr}")
for col in COLOURS:
    for T in [6, 12, 18]:
        for arm in CAL_ARMS:
            h = [agg(arm,col,T,'hidden',x) for x in ('bit_acc','conf','conf_wrong','nll','flag')]
            v = [agg(arm,col,T,'visible',x) for x in ('bit_acc','conf','conf_wrong','nll','flag')]
            fmt = lambda q: (f'{100*q[0]:>11.2f}{q[1]:>7.3f}{100*q[2]:>8.1f}%{q[3]:>8.2f}' +
                             (f'{100*q[4]:>6.1f}%' if not np.isnan(q[4]) else f'{"":>7}'))
            print(f'{col:>6}{EV[(col,T)]["occ"].mean():>7.1f}{arm:>5} | {fmt(h)} | {fmt(v)}')
        print()

print('Per seed detection rate of hidden black cells (flag on hidden cells), arms with an erasure output:')
for arm in ['C', 'F', 'H']:
    print(f'  {arm}: ' + '  '.join(f'{100*np.mean([r["flag"] for r in CAL_ROWS if r["arm"]==arm and r["seed"]==s and r["colour"]=="black" and r["region"]=="hidden"]):5.1f}%'
                                  for s in SEEDS))


                   |                HIDDEN CELLS                |               VISIBLE CELLS               
   col  cells  arm |         acc   conf wrong>.9  -log p   flag |         acc   conf wrong>.9  -log p   flag
  grey    5.7    P |       50.15  0.998    49.6%    6.80        |       88.65  0.998    11.1%    1.47       
  grey    5.7    B |       57.46  0.684     3.0%    0.72        |       94.86  0.944     1.6%    0.16       
  grey    5.7    C |       52.99  0.964    42.1%    3.51  96.6% |       94.96  0.944     1.6%    0.16   1.3%
  grey    5.7    F |       52.46  0.977    44.2%    4.50  98.4% |       95.15  0.944     1.4%    0.15   6.1%
  grey    5.7    H |       52.35  0.966    42.5%    3.56   0.3% |       88.80  0.939     7.7%    0.75   1.9%

  grey   11.5    P |       53.95  0.999    45.8%    6.27        |       86.55  0.998    13.1%    1.75       
  grey   11.5    B |       55.47  0.634     2.0%    0.72        |       93.85  0.944     2.1%    0.18       
  grey   11.5    C

## Structured erasure decoding

The occluder is assumed to be a strip from one side of the marker. 21 hypotheses are tried:
no occlusion, or 1 to 5 cell rows or columns from the top, bottom, left or right. For each
hypothesis, hidden cells are ignored, every id and rotation is scored with soft decoding, and
lambda times the number of ignored cells is subtracted, so ignoring cells is never free. The
best (id, hypothesis) pair wins. With lambda infinite this is plain soft decoding.

lambda is chosen per arm on videos 1 and 2, pooled over seeds and all edge conditions, and
every number in the tables is from videos 3 to 5. `recovered` is the share of the gap between
soft decoding and the oracle that structured decoding closes. `IoU` measures how well the
chosen hypothesis matches the true hidden cells (occluded markers only).


In [17]:
HYP = [np.zeros((6,6), bool)]
for w in range(1, 6):
    for side in range(4):
        m = np.zeros((6,6), bool)
        if side == 0:   m[:w, :] = True
        elif side == 1: m[-w:, :] = True
        elif side == 2: m[:, :w] = True
        else:           m[:, -w:] = True
        HYP.append(m)
HYP = np.stack(HYP).reshape(-1, 36); HSIZE = HYP.sum(1)
LAMBDAS = [0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0, 12.0, float('inf')]
ROT_T = torch.tensor(ROT, device=DEV_)                          # (4, 250, 36)
VIS_T = torch.tensor(~HYP, device=DEV_, dtype=torch.float32)    # (21, 36)
print(f'{len(HYP)} hypotheses, lambda grid {LAMBDAS}')

def structured(P, W=None, chunk=2048):
    """Predicted id and chosen hypothesis for every lambda."""
    P = P.reshape(len(P), 36); out_id = {l: [] for l in LAMBDAS}; out_h = {l: [] for l in LAMBDAS}
    with torch.no_grad():
        for s in range(0, len(P), chunk):
            p = torch.tensor(P[s:s+chunk], device=DEV_, dtype=torch.float32)
            LP, LQ = torch.log(p.clamp(1e-6, 1)), torch.log((1-p).clamp(1e-6, 1))
            if W is not None:
                w = torch.tensor(W[s:s+chunk].reshape(-1, 36), device=DEV_, dtype=torch.float32)
                LP, LQ = LP*w, LQ*w
            best = {l: torch.full((len(p), 250), -float('inf'), device=DEV_) for l in LAMBDAS}
            bh   = {l: torch.zeros((len(p), 250), dtype=torch.long, device=DEV_) for l in LAMBDAS}
            for h in range(len(HYP)):
                a, b = LP*VIS_T[h], LQ*VIS_T[h]
                S = torch.stack([a @ ROT_T[k].T + b @ (1-ROT_T[k]).T for k in range(4)]).amax(0)
                for l in LAMBDAS:
                    if h > 0 and l == float('inf'): continue
                    sc = S - (l*float(HSIZE[h]) if h > 0 else 0.0)
                    upd = sc > best[l]
                    best[l] = torch.where(upd, sc, best[l]); bh[l][upd] = h
            for l in LAMBDAS:
                i = best[l].argmax(1)
                out_id[l].append(i.cpu().numpy()); out_h[l].append(bh[l].gather(1, i[:, None])[:, 0].cpu().numpy())
    return {l: np.concatenate(out_id[l]) for l in LAMBDAS}, {l: np.concatenate(out_h[l]) for l in LAMBDAS}

vid_of = np.array([fr.split('/')[0] for fr in EV[CLEAN_KEY]['frame']])
TUNE = np.isin(vid_of, DEVV[:2]); TEST = ~TUNE
print(f'tuning on {DEVV[:2]} ({TUNE.sum()} markers), testing on {DEVV[2:]} ({TEST.sum()} markers)\n')

STRUCT_ARMS = ['P', 'B', 'E', 'C', 'F', 'H']
VARIANTS = [(a, 'soft') for a in STRUCT_ARMS] + [(a, 'erasure') for a in STRUCT_ARMS
                                                  if a in ARMS and ARMS[a]['out_ch'] == 2]
SC_KEYS = [CLEAN_KEY] + CONDS + CCONDS
ST = {}          # (arm, variant, seed, cond) -> dict
t0 = time.time()
for arm, var in VARIANTS:
    for s in ARM_SEEDS[arm]:
        for k in SC_KEYS:
            d = EV[k]; ids = d['id']; M = d['mask'].reshape(-1, 36)
            P, E = probs(arm, s, k)
            W = (1 - E) if var == 'erasure' else None
            pid, ph = structured(P, W)
            oracle = dec_soft(P, (~M).astype(np.float32) * (W if W is not None else 1.0)) == ids
            occl = M.any(1)
            r = {'oracle_tune': float(oracle[TUNE].mean()), 'oracle_test': float(oracle[TEST].mean())}
            for l in LAMBDAS:
                ok = pid[l] == ids; Hm = HYP[ph[l]]
                inter, union = (Hm & M).sum(1), (Hm | M).sum(1)
                sel = TEST & occl
                r[(l,'tune')] = float(ok[TUNE].mean()); r[(l,'test')] = float(ok[TEST].mean())
                r[(l,'iou')] = float((inter[sel] / np.maximum(union[sel], 1)).mean()) if sel.any() else float('nan')
                r[(l,'none')] = float((ph[l][TEST] == 0).mean())
            ST[(arm, var, s, k)] = r
    print(f'  {arm} {var:<8} done ({time.time()-t0:.0f}s)')

LAM = {}
for arm, var in VARIANTS:
    sc = {l: np.mean([ST[(arm,var,s,k)][(l,'tune')] for s in ARM_SEEDS[arm] for k in CONDS]) for l in LAMBDAS}
    LAM[(arm,var)] = max(sc, key=lambda l: (round(sc[l], 6), l))   # ties go to the larger lambda (closer to plain soft)
    print(f'\n{arm} {var}: tuning accuracy by lambda  ' + '  '.join(f'{l}:{100*v:.2f}' for l, v in sc.items()))
    print(f'{arm} {var}: chosen lambda = {LAM[(arm,var)]}')

def g(arm, var, k, key):
    return 100*np.mean([ST[(arm,var,s,k)][key] for s in ARM_SEEDS[arm]])

print('\n\nTEST VIDEOS ONLY. soft = plain soft decoding (or erasure weighted), struct = structured,')
print('oracle = true hidden cells removed, recovered = share of the soft to oracle gap closed,')
print('IoU = overlap of the chosen hypothesis with the true hidden cells, none = share decoded as unoccluded\n')
STRUCT_TABLE = []
for arm, var in VARIANTS:
    l = LAM[(arm,var)]
    print(f'=== {arm} {var}, lambda {l}')
    print(f"{'cond':>9}{'cells':>7}{'soft':>9}{'struct':>9}{'oracle':>9}{'gain':>8}{'recovered':>11}{'IoU':>7}{'none':>7}")
    for k in SC_KEYS:
        so, st, orc = g(arm,var,k,(float('inf'),'test')), g(arm,var,k,(l,'test')), g(arm,var,k,'oracle_test')
        gap = orc - so; rec = (st - so)/gap if gap > 1 else float('nan')
        iou, none = g(arm,var,k,(l,'iou')), g(arm,var,k,(l,'none'))
        STRUCT_TABLE.append(dict(arm=arm, variant=var, lam=l, cond=k[0], target=k[1],
                                 cells=float(EV[k]['occ'].mean()), soft=so, struct=st, oracle=orc,
                                 gain=st-so, recovered=rec, iou=iou, none=none))
        print(f"{k[0]:>9}{EV[k]['occ'].mean():>7.1f}{so:>9.2f}{st:>9.2f}{orc:>9.2f}{st-so:>8.2f}"
              f"{(f'{100*rec:>10.0f}%' if not np.isnan(rec) else f'{chr(45):>11}')}{iou:>7.1f}{none:>6.1f}%")
    print()


21 hypotheses, lambda grid [0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0, 12.0, inf]
tuning on ['video_1', 'video_2'] (4938 markers), testing on ['video_3', 'video_4', 'video_5'] (7338 markers)

  P soft     done (146s)
  B soft     done (209s)
  E soft     done (272s)
  C soft     done (334s)
  F soft     done (396s)
  H soft     done (458s)
  C erasure  done (521s)
  F erasure  done (583s)
  H erasure  done (644s)

P soft: tuning accuracy by lambda  0.1:59.48  0.25:60.39  0.5:61.11  0.75:61.89  1.0:63.54  1.5:66.16  2.0:67.34  3.0:71.81  4.0:70.29  6.0:65.30  8.0:61.59  12.0:58.91  inf:58.90
P soft: chosen lambda = 3.0

B soft: tuning accuracy by lambda  0.1:64.54  0.25:79.12  0.5:86.54  0.75:86.27  1.0:84.80  1.5:83.09  2.0:82.10  3.0:81.70  4.0:81.58  6.0:81.42  8.0:81.40  12.0:81.40  inf:81.40
B soft: chosen lambda = 0.5

E soft: tuning accuracy by lambda  0.1:60.07  0.25:79.07  0.5:87.92  0.75:87.31  1.0:85.62  1.5:83.85  2.0:83.26  3.0:83.05  4.0:82.92  6.0:82.86  8.0

## Paired comparisons

Arms with the same number of outputs start from the same weights and see the same batches in
the same order, so seed s of one arm is the natural partner of seed s of the other. The
interval is a paired t interval over the 5 seed differences. `differs` means the interval
excludes zero; `equivalent` means it lies inside plus or minus 1 point; otherwise
`inconclusive`. The last block tests the structured decoding gain within each arm, which is
paired by construction (same model, same markers).


In [18]:
def paired_t(a, b):
    d = np.asarray(a, float) - np.asarray(b, float); m = d.mean()
    if len(d) < 2 or d.std(ddof=1) == 0: return float(m), float(m), float(m)
    h = sst.t.ppf(0.975, len(d)-1) * d.std(ddof=1) / np.sqrt(len(d))
    return float(m), float(m-h), float(m+h)

def verdict(lo, hi):
    if lo > 0 or hi < 0: return 'differs'
    if lo > -MARGIN and hi < MARGIN: return f'equivalent (within {MARGIN})'
    return 'inconclusive'

PAIRS = [('E','soft','B','soft','random brightness training'),
         ('G','soft','M','soft','average against max pooling'),
         ('F','soft_erasure','C','soft_erasure','random brightness, erasure arms'),
         ('H','soft_erasure','C','soft_erasure','black only against grey, erasure arms'),
         ('H','soft_erasure','F','soft_erasure','black only against random, erasure arms'),
         ('D','soft_erasure','C','soft_erasure','unmasked against masked loss, erasure arms')]
PAIRED = []
for a1, d1, a2, d2, q in PAIRS:
    assert ARMS[a1]['out_ch'] == ARMS[a2]['out_ch'], 'pairing across groups is not valid'
    print(f'=== {a1} ({d1}) minus {a2} ({d2}): {q}')
    print(f"{'col':>7}{'cells':>7}{'diff':>9}{'95% CI':>20}  verdict")
    for col in COLOURS:
        for T in TARGETS:
            m, lo, hi = paired_t(seed_values(a1,col,T,d1), seed_values(a2,col,T,d2)); v = verdict(lo, hi)
            PAIRED.append(dict(a1=a1, d1=d1, a2=a2, d2=d2, colour=col, target=T, diff=m, lo=lo, hi=hi, verdict=v))
            print(f'{col:>7}{EV[(col,T)]["occ"].mean():>7.1f}{m:>9.2f}   [{lo:>6.2f},{hi:>6.2f}]  {v}')
    print()

print('=== structured minus plain decoding, same model, test videos, paired over seeds')
print(f"{'arm':>4}{'variant':>9}{'cond':>9}{'cells':>7}{'gain':>9}{'95% CI':>20}  verdict")
for arm, var in VARIANTS:
    if arm == 'P': continue
    l = LAM[(arm,var)]
    for k in [CLEAN_KEY] + [(c, T) for c in COLOURS for T in (6, 12, 18)] + CCONDS:
        a = [100*ST[(arm,var,s,k)][(l,'test')] for s in SEEDS]
        b = [100*ST[(arm,var,s,k)][(float('inf'),'test')] for s in SEEDS]
        m, lo, hi = paired_t(a, b); v = verdict(lo, hi)
        PAIRED.append(dict(a1=arm, d1=f'struct_{var}', a2=arm, d2=var, colour=k[0], target=k[1], diff=m, lo=lo, hi=hi, verdict=v))
        print(f'{arm:>4}{var:>9}{k[0]:>9}{EV[k]["occ"].mean():>7.1f}{m:>9.2f}   [{lo:>6.2f},{hi:>6.2f}]  {v}')
    print()


=== E (soft) minus B (soft): random brightness training
    col  cells     diff              95% CI  verdict
   grey    3.2     0.26   [ -1.34,  1.86]  inconclusive
   grey    5.7     1.01   [ -1.40,  3.42]  inconclusive
   grey    8.5     1.26   [ -1.48,  4.00]  inconclusive
   grey   11.5     1.28   [ -0.76,  3.32]  inconclusive
   grey   15.6     1.56   [ -1.25,  4.37]  inconclusive
   grey   18.0     1.95   [  0.15,  3.75]  differs
  white    3.2     1.44   [ -3.06,  5.94]  inconclusive
  white    5.7     2.50   [ -2.17,  7.16]  inconclusive
  white    8.5     1.95   [ -2.44,  6.33]  inconclusive
  white   11.5     1.84   [ -1.88,  5.56]  inconclusive
  white   15.6     1.90   [ -1.78,  5.59]  inconclusive
  white   18.0     2.03   [ -0.67,  4.73]  inconclusive
  black    3.2     1.71   [ -2.12,  5.54]  inconclusive
  black    5.7     2.39   [ -3.55,  8.34]  inconclusive
  black    8.5     3.21   [ -3.87, 10.29]  inconclusive
  black   11.5     0.99   [ -6.52,  8.49]  inconclusive


In [19]:
res = dict(code=CODE_TAG, seeds=SEEDS, margin=MARGIN,
           arms={k: {kk: str(vv) for kk,vv in v.items()} for k,v in ARMS.items()},
           levels={str(k): v for k,v in LEVEL_FOR.items()}, centre_levels={str(k): v for k,v in LEVEL_C.items()},
           accuracy_soft={f'{a}|{s}|{c}|{T}': float(ACC[(a,s,c,T)].mean()) for (a,s,c,T) in ACC},
           lambdas={f'{a}|{v}': (None if l == float('inf') else l) for (a,v), l in LAM.items()},
           structured=STRUCT_TABLE, paired=PAIRED, calibration=CAL_ROWS,
           structured_by_seed=[dict(arm=a, variant=v, seed=s, cond=k[0], target=k[1],
                                    **{('soft' if l == float('inf') else f'lam_{l}')+'_'+sp: val
                                       for (l, sp), val in [(kk, vv) for kk, vv in r.items() if isinstance(kk, tuple)]},
                                    oracle_tune=r['oracle_tune'], oracle_test=r['oracle_test'])
                               for (a, v, s, k), r in ST.items()])
with open(f'{ART}/n15_dev.json','w') as fh: json.dump(res, fh, indent=1)
print('saved', f'{ART}/n15_dev.json')


saved /content/drive/MyDrive/aruco_p2/artifacts/n15_dev.json


## Results workbook

Collects every `.json` result file in the artifacts folder into `all_results.xlsx` (saved in
the same folder, and downloaded automatically in Colab). One tab per file, an INDEX tab
listing every file with its notebook, date and size, and a COVERAGE tab showing which of
N0 to N15 have nothing saved. This cell only needs the folder paths, so it can be rerun on
its own after future notebooks; the separate export notebook does exactly that.


In [20]:
# ---------------------------------------------------------------------------------
# Collect every saved result (all .json files in the artifacts folder) into one Excel
# workbook. One tab per file, plus an INDEX tab listing every file and a COVERAGE tab
# showing which notebooks have no saved results yet. Safe to rerun at any time.
# ---------------------------------------------------------------------------------
import re, glob, json, math, datetime
import numpy as np, pandas as pd
from openpyxl.styles import Font

XLSX = f'{ART}/all_results.xlsx'
NOTEBOOKS = list(range(0, 16))          # notebooks expected to have saved something
MAX_ROWS = 1_000_000

# how to name the parts of keys written as 'a|b|c'
KNOWN = {
    'accuracy':        ['arm', 'seed', 'colour', 'target'],
    'accuracy_soft':   ['arm', 'seed', 'colour', 'target'],
    'seconds':         ['arm', 'seed'],
    'cap':             ['seed', 'eps', 'colour', 'target', 'split'],
    'n13_unified_dev': ['arm', 'seed', 'colour', 'target', 'decoder'],
    'n14_unified_dev': ['arm', 'seed', 'colour', 'target', 'decoder'],
    'n15_unified_dev': ['arm', 'seed', 'colour', 'target', 'decoder'],
}
NOTES = {   # results that later notebooks replaced
    'n13_seeds_dev':   'superseded by N14 (N13 mixed checkpoints from two code versions)',
    'n13_unified_dev': 'superseded by N14 (N13 mixed checkpoints from two code versions)',
}

def is_scalar(x): return x is None or isinstance(x, (str, int, float, bool))

def num(s):
    try: return pd.to_numeric(s)
    except (ValueError, TypeError): return s

def cols_for(name, stem, n):
    for part in [p for p in reversed(name.split('.'))] + [stem]:
        if part in KNOWN and len(KNOWN[part]) == n: return KNOWN[part]
    return [f'key{i+1}' for i in range(n)]

def split_keys(d, name, stem):
    rows = [str(k).split('|') for k in d]
    n = max(len(r) for r in rows)
    names = cols_for(name, stem, n)
    df = pd.DataFrame([r + [None]*(n-len(r)) for r in rows], columns=names)
    for c in names: df[c] = num(df[c])
    df['value'] = list(d.values())
    return df

def tables(obj, name, stem):
    """Yield (title, DataFrame) pairs for any JSON value."""
    if isinstance(obj, list):
        if obj and all(isinstance(x, dict) for x in obj):
            yield name, pd.json_normalize(obj, sep='.')
        elif all(is_scalar(x) for x in obj):
            yield name, pd.DataFrame({name.split('.')[-1] or 'value': obj})
        else:
            yield name, pd.DataFrame(obj)
        return
    if not isinstance(obj, dict):
        yield name, pd.DataFrame({'value': [obj]}); return

    scal = {k: v for k, v in obj.items() if is_scalar(v)}
    rest = {k: v for k, v in obj.items() if not is_scalar(v)}
    if scal:
        if len(scal) >= 3 and all('|' in str(k) for k in scal):
            yield name, split_keys(scal, name, stem)
        else:
            yield name, pd.DataFrame({'key': [str(k) for k in scal], 'value': list(scal.values())})
    # a dict of small flat dicts (e.g. arm settings) becomes one table, one row each
    flat_dicts = {k: v for k, v in rest.items() if isinstance(v, dict) and v
                  and all(is_scalar(x) for x in v.values()) and not any('|' in str(x) for x in v)}
    if len(flat_dicts) >= 2:
        df = pd.DataFrame.from_dict(flat_dicts, orient='index')
        df.insert(0, 'name', df.index.astype(str)); yield name, df.reset_index(drop=True)
        rest = {k: v for k, v in rest.items() if k not in flat_dicts}
    for k, v in rest.items():
        yield from tables(v, f'{name}.{k}' if name else str(k), stem)

def clean(df):
    df = df.copy()
    for c in df.columns:
        if df[c].map(lambda x: isinstance(x, (list, dict))).any():
            df[c] = df[c].map(lambda x: json.dumps(x) if isinstance(x, (list, dict)) else x)
        if df[c].dtype.kind == 'f':
            df[c] = df[c].where(np.isfinite(df[c]), None)
    df.columns = [str(c) for c in df.columns]
    return df.head(MAX_ROWS)

def sheet_name(stem, used):
    s = re.sub(r'[\[\]:*?/\\]', '_', stem)[:31] or 'sheet'
    base, i = s, 2
    while s.lower() in used:
        s = f'{base[:28]}_{i}'; i += 1
    used.add(s.lower()); return s

files = sorted(set(glob.glob(f'{ART}/**/*.json', recursive=True)) | set(glob.glob(f'{BASE}/*.json')))
def nb_of(path):
    m = re.match(r'^n(\d+)', os.path.basename(path).lower())
    return int(m.group(1)) if m else None
files.sort(key=lambda p: (nb_of(p) if nb_of(p) is not None else -1, os.path.basename(p)))
print(f'found {len(files)} result files under {ART}\n')

index, used, bad = [], {'index', 'coverage'}, []
with pd.ExcelWriter(XLSX, engine='openpyxl') as xw:
    pd.DataFrame({'building': ['...']}).to_excel(xw, sheet_name='INDEX', index=False)
    pd.DataFrame({'building': ['...']}).to_excel(xw, sheet_name='COVERAGE', index=False)
    for path in files:
        stem = os.path.splitext(os.path.basename(path))[0]
        try:
            obj = json.load(open(path))
            parts = [(t, clean(df)) for t, df in tables(obj, '', stem) if len(df.columns)]
        except Exception as e:
            bad.append((path, repr(e))); print(f'  could not read {path}: {e!r}'); continue
        sh = sheet_name(stem, used); row = 0
        for title, df in parts:
            df.to_excel(xw, sheet_name=sh, startrow=row + 1, index=False)
            ws = xw.sheets[sh]
            ws.cell(row=row + 1, column=1, value=title or stem).font = Font(bold=True)
            row += len(df) + 4
        index.append(dict(notebook=f'N{nb_of(path)}' if nb_of(path) is not None else 'setup',
                          sheet=sh, file=os.path.relpath(path, BASE),
                          modified=datetime.datetime.fromtimestamp(os.path.getmtime(path)).strftime('%Y-%m-%d %H:%M'),
                          size_kb=round(os.path.getsize(path)/1024, 1), tables=len(parts),
                          rows=int(sum(len(d) for _, d in parts)), note=NOTES.get(stem, '')))
        print(f'  {sh:<32} {len(parts):>3} tables {index[-1]["rows"]:>8} rows   {index[-1]["file"]}')

    idx = pd.DataFrame(index)
    have = {nb_of(p) for p in files if nb_of(p) is not None}
    cov = pd.DataFrame([dict(notebook=f'N{n}', files=int(sum(1 for p in files if nb_of(p) == n)),
                             status='saved' if n in have else
                                    'NOTHING SAVED: rerun this notebook, or its results exist only as printed output')
                        for n in NOTEBOOKS])
    idx.to_excel(xw, sheet_name='INDEX', index=False)
    cov.to_excel(xw, sheet_name='COVERAGE', index=False)
    for name in ('INDEX', 'COVERAGE'):
        ws = xw.sheets[name]
        for cell in ws[1]: cell.font = Font(bold=True)
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = min(80, max(10, max(len(str(c.value or '')) for c in col) + 2))

print(f'\nsaved {XLSX}')
print('\nCOVERAGE')
print(cov.to_string(index=False))
if bad: print('\nfiles that could not be read:', *bad, sep='\n  ')
try:
    from google.colab import files as _files
    _files.download(XLSX)
except Exception:
    pass


found 23 result files under /content/drive/MyDrive/aruco_p2/artifacts

  could not read /content/drive/MyDrive/aruco_p2/artifacts/manifest.json: ValueError('cannot insert name, already exists')
  splits                             3 tables        8 rows   artifacts/splits.json
  n1a_headroom                       3 tables       23 rows   artifacts/n1a_headroom.json
  n2_occlusion                       6 tables       32 rows   artifacts/n2_occlusion.json
  n3_results                         4 tables       21 rows   artifacts/n3_results.json
  n3b_results                        3 tables       23 rows   artifacts/n3b_results.json
  n4_controlled                    188 tables      506 rows   artifacts/n4_controlled.json
  n5_significance                    4 tables      109 rows   artifacts/n5_significance.json
  n6_mechanism_mitigation            4 tables       20 rows   artifacts/n6_mechanism_mitigation.json
  n7_error_localisation              4 tables       23 rows   artifacts/n7_error

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>